In [1]:
import random

In [2]:
import random
import math

class CBMInstanceGenerator:
    def __init__(self,
                 lines: int,
                 cols: int,
                 density: float,
                 output_file: str = "instance.txt",
                 seed: int | None = None,
                 variance: float = 0.001):
        """
        lines       -> number of rows
        cols        -> number of columns
        density     -> target fraction of 1s per column (0–1), e.g. 0.05 for 5%
        variance    -> allowed spread around density (default 0.001 = ±0.1%)
        output_file -> path to save instance
        seed        -> optional RNG seed for reproducibility
        """
        if not (0 <= density <= 1):
            raise ValueError("density must be in [0, 1]")
        if variance < 0:
            raise ValueError("variance must be >= 0")

        self.lines = lines
        self.cols = cols
        self.density = density
        self.variance = variance
        self.output_file = output_file
        self.random = random.Random(seed)
        self.matrix = None

    # ---------- generation logic ----------

    def _ones_for_column(self) -> int:
        """
        Sample the number of 1s for one column from a narrow window
        [density - variance, density + variance], clamped to [0, 1].
        The window is converted to an integer range in [0, lines].
        """
        lo = max(0.0, self.density - self.variance)
        hi = min(1.0, self.density + self.variance)
        min_ones = math.floor(lo * self.lines)
        max_ones = math.ceil(hi * self.lines)

        # Guarantee at least 1 one when density > 0
        if self.density > 0:
            min_ones = max(min_ones, 1)
            max_ones = max(max_ones, 1)

        min_ones = min(min_ones, self.lines)
        max_ones = min(max_ones, self.lines)

        return self.random.randint(min_ones, max_ones)

    def _generate_matrix(self):
        matrix = [[0] * self.cols for _ in range(self.lines)]
        for col in range(self.cols):
            k = self._ones_for_column()
            for r in self.random.sample(range(self.lines), k):
                matrix[r][col] = 1
        return matrix

    def generate(self, store_matrix: bool = False):
        matrix = self._generate_matrix()
        if store_matrix:
            self.matrix = matrix
        self._write_file(matrix)

    def _write_file(self, matrix):
        with open(self.output_file, "w") as f:
            f.write(f"{self.lines} {self.cols}\n")
            for row in matrix:
                ones = [str(i + 1) for i, v in enumerate(row) if v == 1]
                f.write(str(len(ones)))
                if ones:
                    f.write(" " + " ".join(ones))
                f.write("\n")

    def stats(self):
        if self.matrix is None:
            print("Matrix not stored. Run generate(store_matrix=True).")
            return
        col_counts = [sum(self.matrix[r][c] for r in range(self.lines))
                      for c in range(self.cols)]
        densities = [c / self.lines for c in col_counts]
        print(f"Target density : {self.density:.4f}")
        print(f"Variance window: ±{self.variance:.4f}")
        print(f"Min col density: {min(densities):.4f}")
        print(f"Max col density: {max(densities):.4f}")
        print(f"Avg col density: {sum(densities) / self.cols:.4f}")

In [3]:
gen = CBMInstanceGenerator(lines=1000, cols=500, density=0.05, variance=0.001, seed=42)
gen.generate(store_matrix=True)
gen.stats()
# Target density : 0.0500
# Variance window: ±0.0010
# Min col density: 0.0490
# Max col density: 0.0510
# Avg col density: 0.0500

Target density : 0.0500
Variance window: ±0.0010
Min col density: 0.0490
Max col density: 0.0520
Avg col density: 0.0504


In [4]:
lines = [1100, 1200, 1300, 1400, 1500]
cols = [11000, 12000, 13000, 14000, 15000]
density = [0.02, 0.05, 0.1, 0.15]
variance = 0.005

for l, c, d in [(l, c, d) for l in lines for c in cols for d in density]:
    filename = f"/home/pedroldm/MSc/cbm/instances/cbm_{l}_{c}_{int(d*100)}.txt"
    print(f"Generating {filename}...")
    gen = CBMInstanceGenerator(lines=l, cols=c, density=d, variance=variance, seed=42, output_file=filename)
    gen.generate()

Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_11000_2.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_11000_5.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_11000_10.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_11000_15.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_12000_2.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_12000_5.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_12000_10.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_12000_15.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_13000_2.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_13000_5.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_13000_10.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_13000_15.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_14000_2.txt...
Generating /home/pedroldm/MSc/cbm/instances/cbm_1100_14000_5.txt...
Generating /home/pedroldm/MSc/cbm/instance